In [1]:
import pandas as pd

customers_df = pd.DataFrame({
    "customer_id":[1,2,3,4,5,6,7,8],
    "name":["Alice","Bob","Charlie","Diana","Eve","Frank","Grace","Henry"],
    "city":["London","Paris","London","Berlin","Paris","London","Berlin","London"],
    "signup_year":[2022,2021,2023,2020,2022,2021,2023,2022]
})

restaurants_df = pd.DataFrame({
    "restaurant_id":[101,102,103,104,105],
    "restaurant_name":[
        "Pizza Palace","Sushi World","Burger Hub",
        "Pasta Corner","Taco Town"
    ],
    "cuisine":[
        "Italian","Japanese","American",
        "Italian","Mexican"
    ],
    "city":[
        "London","Paris","London",
        "Berlin","Paris"
    ]
})
orders_df = pd.DataFrame({
    "order_id":[
        1001,1002,1003,1004,1005,1006,
        1007,1008,1009,1010,1011,1012
    ],
    "customer_id":[
        1,2,3,1,4,5,
        6,2,3,7,8,1
    ],
    "restaurant_id":[
        101,102,103,104,105,101,
        102,103,104,105,101,102
    ],
    "items_count":[
        2,3,1,4,2,1,
        2,3,2,1,5,2
    ],
    "order_time_min":[
        30,25,20,35,28,22,
        26,24,29,21,40,27
    ],
    "order_date":pd.to_datetime([
        "2024-02-01","2024-02-02","2024-02-02",
        "2024-02-03","2024-02-04","2024-02-04",
        "2024-02-05","2024-02-06","2024-02-07",
        "2024-02-08","2024-02-08","2024-02-09"
    ])
})

customers_df = customers_df.rename(columns={'city':'customer_city'})
restaurants_df = restaurants_df.rename(columns={'city':'rester_city'})

In [2]:
'''
Task 1 - Build a master dataset
'''

master_df = (
    customers_df
    .merge(orders_df, on='customer_id', how='left')
    .merge(restaurants_df, on='restaurant_id', how='left')
    .assign(large_order = lambda x: x['items_count'] >= 3)
)
master_df.head()

,customer_id,name,customer_city,signup_year,order_id,restaurant_id,items_count,order_time_min,order_date,restaurant_name,cuisine,rester_city,large_order
0,1,Alice,London,2022,1001,101,2,30,2024-02-01,Pizza Palace,Italian,London,False
1,1,Alice,London,2022,1004,104,4,35,2024-02-03,Pasta Corner,Italian,Berlin,True
2,1,Alice,London,2022,1012,102,2,27,2024-02-09,Sushi World,Japanese,Paris,False
3,2,Bob,Paris,2021,1002,102,3,25,2024-02-02,Sushi World,Japanese,Paris,True
4,2,Bob,Paris,2021,1008,103,3,24,2024-02-06,Burger Hub,American,London,True


In [3]:
'''
Task 2 - Identify active customers (customer = active, when atleast 2 order are made)
'''
active_customers_df = (
    master_df
    .groupby('customer_id')
    .agg(number_of_orders = ('order_id','nunique'))
    .loc[lambda o : o['number_of_orders'] >=2]
)
active_customers_df

,number_of_orders
customer_id,
1,3
2,2
3,2


In [4]:
'''
Task 3 - Cuisine popularity by city
'''

city_cuisine_popularity_df = (
    master_df
    .groupby(['customer_city','cuisine'])
    .agg(
        number_of_orders = ('order_id','nunique'),
        average_items_per_order = ('items_count','mean'),
        average_order_time = ('order_time_min','mean'),
    )
    .reset_index()
)
city_cuisine_popularity_df

,customer_city,cuisine,number_of_orders,average_items_per_order,average_order_time
0,Berlin,Mexican,2,1.50,24.5
1,London,American,1,1.00,20.0
2,London,Italian,4,3.25,33.5
3,London,Japanese,2,2.00,26.5
4,Paris,American,1,3.00,24.0
5,Paris,Italian,1,1.00,22.0
6,Paris,Japanese,1,3.00,25.0


In [5]:
'''
Task 4 - Top cuisine per city
'''

top_cuisine_per_city_df = (
    city_cuisine_popularity_df
    .loc[lambda x: x['number_of_orders'] == x.groupby('customer_city')['number_of_orders'].transform('max')]
)
top_cuisine_per_city_df

,customer_city,cuisine,number_of_orders,average_items_per_order,average_order_time
0,Berlin,Mexican,2,1.50,24.5
2,London,Italian,4,3.25,33.5
4,Paris,American,1,3.00,24.0
5,Paris,Italian,1,1.00,22.0
6,Paris,Japanese,1,3.00,25.0
